In [0]:
# Databricks notebook source
# SILVER LAYER: Circuit Dimension Table
# Source: meetings (richest circuit data — has circuit_type, images, etc.)

from pyspark.sql.functions import (
    col, trim, upper, lower, initcap, when, lit,
    current_timestamp, coalesce, to_timestamp,
    datediff, round as spark_round, row_number
)
from pyspark.sql.window import Window
from pyspark.sql import functions as F

# ============ 1. READ FROM BRONZE ============
#Derive the tables you will need to complete the task as variables

meetings_raw = spark.read.table("f1_data.bronze.meetings")
sessions_raw = spark.read.table("f1_data.bronze.sessions")

print(f"Bronze meetings: {meetings_raw.count()}")
print(f"Bronze sessions: {sessions_raw.count()}")

# ============ 2. CLEAN MEETINGS ============
meetings_clean = meetings_raw \
    .withColumn("meeting_name", initcap(trim(col("meeting_name")))) \
    .withColumn("meeting_official_name", trim(col("meeting_official_name"))) \
    .withColumn("circuit_short_name", trim(col("circuit_short_name"))) \
    .withColumn("circuit_type",
        when(col("circuit_type").isNotNull(), upper(trim(col("circuit_type"))))
        .otherwise(lit("UNKNOWN"))
    ) \
    .withColumn("country_name", initcap(trim(col("country_name")))) \
    .withColumn("country_code", upper(trim(col("country_code")))) \
    .withColumn("location", initcap(trim(col("location")))) \
    .withColumn("is_cancelled", coalesce(col("is_cancelled"), lit(False))) \
    .withColumn("year", col("year").cast("integer")) \
    .withColumn("meeting_duration_days",
        datediff(col("date_end"), col("date_start"))
    )

# ============ 3. CLEAN SESSIONS ============
sessions_clean = sessions_raw \
    .withColumn("session_name", trim(col("session_name"))) \
    .withColumn("session_type",
        when(col("session_type").isNotNull(), initcap(trim(col("session_type"))))
        .otherwise(lit("Unknown"))
    ) \
    .withColumn("location", initcap(trim(col("location")))) \
    .withColumn("country_code", upper(trim(col("country_code")))) \
    .withColumn("country_name", initcap(trim(col("country_name")))) \
    .withColumn("gmt_offset", trim(col("gmt_offset"))) \
    .withColumn("is_cancelled", coalesce(col("is_cancelled"), lit(False))) \
    .withColumn("session_duration_hours",
        spark_round(
            (col("date_end").cast("long") - col("date_start").cast("long")) / 3600, 2
        )
    )

# ============ 4. VALIDATE DATE ORDER ============
# date_start must be before date_end for both tables
invalid_meetings = meetings_clean.filter(col("date_start") > col("date_end"))
invalid_sessions = sessions_clean.filter(col("date_start") > col("date_end"))

if invalid_meetings.count() > 0 or invalid_sessions.count() > 0:
    print(f"⚠️  {invalid_meetings.count()} meetings with invalid date order")
    print(f"⚠️  {invalid_sessions.count()} sessions with invalid date order")
    
    # If there are any invalid meetings or sessions, make sure to save these records into a new table.
    invalid_meetings.withColumn("rejection_reason", lit("date_start > date_end")) \
        .write.mode("append").format("delta").saveAsTable("f1_data.silver._rejected_records")

# Remove date-invalid records before proceeding
meetings_clean = meetings_clean.filter(
    col("date_start").isNull() | col("date_end").isNull() | 
    (col("date_start") <= col("date_end"))
)
sessions_clean = sessions_clean.filter(
    col("date_start").isNull() | col("date_end").isNull() | 
    (col("date_start") <= col("date_end"))
)

# ============ 5. DEDUPLICATE MEETINGS ============
meetings_dedup_window = Window.partitionBy("meeting_key").orderBy(col("date_start").desc())

meetings_deduped = meetings_clean \
    .withColumn("row_num", row_number().over(meetings_dedup_window)) \
    .filter(col("row_num") == 1) \
    .drop("row_num")

# ============ 6. JOIN SESSIONS TO MEETINGS (enrich with circuit context) ============
# sessions has circuit_key + meeting_key; meetings has full circuit detail
sessions_enriched = sessions_clean.join(
    meetings_deduped.select(
        "meeting_key", "meeting_name", "meeting_official_name",
        "circuit_type", "circuit_info_url", "country_flag",
        "meeting_duration_days"
    ),
    on="meeting_key",
    how="left"
)

# ============ 7. ADD SILVER METADATA ============
sessions_silver = sessions_enriched \
    .withColumn("silver_processed_at", current_timestamp()) \
    .withColumn("silver_source", lit("f1_data.bronze.sessions + meetings")) \
    .withColumn("data_quality_flag", lit("PASSED"))

meetings_silver = meetings_deduped \
    .withColumn("silver_processed_at", current_timestamp()) \
    .withColumn("silver_source", lit("f1_data.bronze.meetings")) \
    .withColumn("data_quality_flag", lit("PASSED"))

# ============ 8. SELECT FINAL COLUMNS ============
meetings_silver = meetings_silver.select(
    "meeting_key",
    "meeting_name",
    "meeting_official_name",
    "circuit_key",
    "circuit_short_name",
    "circuit_type",
    "circuit_image",
    "circuit_info_url",
    "country_key",
    "country_code",
    "country_name",
    "country_flag",
    "location",
    "date_start",
    "date_end",
    "gmt_offset",
    "year",
    "is_cancelled",
    "meeting_duration_days",
    "data_quality_flag",
    "silver_processed_at",
    "silver_source"
)

sessions_silver = sessions_silver.select(
    "session_key",
    "meeting_key",
    "session_name",
    "session_type",
    "circuit_key",
    "circuit_short_name",
    "country_code",
    "country_key",
    "country_name",
    "location",
    "date_start",
    "date_end",
    "gmt_offset",
    "year",
    "is_cancelled",
    "session_duration_hours",
    "meeting_name",
    "meeting_official_name",
    "circuit_type",
    "data_quality_flag",
    "silver_processed_at",
    "silver_source"
)

# ============ 9. WRITE TO SILVER ============
meetings_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year") \
    .saveAsTable("f1_data.silver.dim_meetings")

sessions_silver.write \
    .mode("overwrite") \
    .format("delta") \
    .partitionBy("year", "session_type") \
    .saveAsTable("f1_data.silver.dim_sessions")

print(f"✓ Silver dim_meetings written: {meetings_silver.count()} records")
print(f"✓ Silver dim_sessions written: {sessions_silver.count()} records")

Bronze meetings: 25
Bronze sessions: 123
✓ Silver dim_meetings written: 25 records
✓ Silver dim_sessions written: 123 records
